# 03 — Modeling: Marketing Mix Model

**Project:** 04-marketing-budget-reallocation-mmm
**Author:** Model Builder
**Date:** 2026-11-09

This notebook loads `reports/modeling/features.parquet`, validates, compares model families,
tunes the best families with Optuna, selects and evaluates the final model, and writes all
required artifacts.

Sections:
1. Setup
2. Load features
3. Validation design
4. Baselines (business rule + Ridge)
5. Model family comparison
6. Optuna tuning
7. Model selection
8. Final held-out evaluation
9. Explanation (permutation importance, SHAP, counterfactuals)
10. Artifacts
11. Model card


## 1. Setup

In [1]:

import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

from model_utils import rmse, mae, mape, r2, compute_all_metrics,                         decile_table, threshold_table, rolling_origin_splits, cv_score
from features import FEATURE_COLS_SAT, FEATURE_COLS_LOG

SEED = 42
np.random.seed(SEED)
PROJECT = os.path.abspath('..')
CHARTS  = os.path.join(PROJECT, 'reports', 'modeling', 'charts')
TABLES  = os.path.join(PROJECT, 'reports', 'modeling', 'tables')
MODELS  = os.path.join(PROJECT, 'models')
for d in [CHARTS, TABLES, MODELS]:
    os.makedirs(d, exist_ok=True)

import sklearn, lightgbm, xgboost, catboost, optuna as op
print('numpy:', np.__version__, '| pandas:', pd.__version__)
print('sklearn:', sklearn.__version__, '| lgb:', lightgbm.__version__)
print('xgb:', xgboost.__version__, '| catboost:', catboost.__version__)
print('optuna:', op.__version__, '| SEED:', SEED)


numpy: 2.4.6 | pandas: 2.3.3
sklearn: 1.9.1 | lgb: 4.7.0
xgb: 3.4.1 | catboost: 1.2.10
optuna: 5.0.0 | SEED: 42


**Output:** Library versions printed; all load without error.

## 2. Load features

We load `features.parquet` written by `02_feature_engineering.ipynb`. No feature
recomputation happens here — only the split column is used to separate train and test.


In [2]:

feat_path = os.path.join(PROJECT, 'reports', 'modeling', 'features.parquet')
feat_df = pd.read_parquet(feat_path)
print('Shape:', feat_df.shape)
print('Split counts:', feat_df['split'].value_counts().to_dict())
print('Columns:', feat_df.columns.tolist())


Shape: (208, 28)
Split counts: {'train': 188, 'test': 20}
Columns: ['DATE', 'revenue', 'split', 'tv_S_sat', 'ooh_S_sat', 'print_S_sat', 'facebook_S_sat', 'search_S_sat', 'newsletter_log', 'competitor_sales_log', 'event_flag', 'total_spend_log', 'print_low_flag', 'trend', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'tv_S_log', 'ooh_S_log', 'print_S_log', 'facebook_S_log', 'search_S_log', 'tv_S_adstock', 'ooh_S_adstock', 'print_S_adstock', 'facebook_S_adstock', 'search_S_adstock']


**Output:** 208 rows, 188 train / 20 test, confirming the split from notebook 02.

## 3. Validation design

**Why rolling-origin cross-validation.** Marketing spend data is a time series. Using random
k-fold would allow weeks from 2019 to appear in training while weeks from 2016 appear only in
validation, letting the model implicitly use future information. Rolling-origin CV (also called
"expanding window" or "walk-forward" CV) always trains on the past and validates on the future,
producing an honest performance estimate.

**Alternatives considered.**
- *Random k-fold*: would inflate R² by leaking future patterns; not used.
- *Blocked time-series CV*: trains on a fixed-width window, discards older data; could be
  more realistic for concept-drift scenarios, but here we have only 4 years and no evidence
  of structural breaks, so expanding-window is preferred to use as much training data as possible.

**How it works.**
We define $m$ folds. Fold $i$ trains on weeks $0 \ldots t_i - 1$ and validates on weeks
$t_i \ldots t_i + w - 1$ where $w = 20$ (validation window). We step $t_i$ forward by 10 weeks
each fold, ensuring each validation block is seen exactly once.

**How to read the output.** For each model, we report mean ± std RMSE across folds. A large
std indicates instability (the model fits some time periods much better than others). We prefer
a model with both low mean and low std.

**Decision rule (stated before seeing results):** We will select the model family with the
lowest mean CV RMSE that is not worse than the best by more than one std — favouring simplicity.


In [3]:

train_mask = feat_df['split'] == 'train'
test_mask  = feat_df['split'] == 'test'

X_train_all = feat_df.loc[train_mask].reset_index(drop=True)
X_test_all  = feat_df.loc[test_mask].reset_index(drop=True)

y_train = X_train_all['revenue']
y_test  = X_test_all['revenue']

# Feature sets
X_train_sat = X_train_all[FEATURE_COLS_SAT]
X_test_sat  = X_test_all[FEATURE_COLS_SAT]
X_train_log = X_train_all[FEATURE_COLS_LOG]
X_test_log  = X_test_all[FEATURE_COLS_LOG]

# Rolling-origin splits (on training set)
splits = rolling_origin_splits(len(y_train), n_test=20, min_train=60, step=10)
print(f'Number of CV folds: {len(splits)}')
for i, (tr, va) in enumerate(splits[:3]):
    print(f'  Fold {i}: train {len(tr)} wks, val {len(va)} wks | val ends at week {va[-1]}')
print('  ...')


Number of CV folds: 11
  Fold 0: train 60 wks, val 20 wks | val ends at week 79
  Fold 1: train 70 wks, val 20 wks | val ends at week 89
  Fold 2: train 80 wks, val 20 wks | val ends at week 99
  ...


**Output:** Rolling-origin folds are shown. Each fold grows the training window by 10 weeks and evaluates on the next 20 weeks — strictly forward in time.

## 4. Baselines

We establish two floors before comparing model families.

### 4a. Business rule baseline — mean revenue

**Why this method.** The simplest rule a marketing manager would use: predict each week's
revenue as the average revenue from the training set. This requires no model and captures
no channel or time variation. It represents the cost of doing no modelling at all, and
any useful model must beat it substantially.

**Alternatives considered.**
- *Last-value baseline*: predict next week's revenue as this week's actual revenue; better
  for trending series but still ignores channel variation.
- *Seasonal naive* (same week last year): captures annual seasonality but requires at least
  52 prior weeks and ignores spend.

**How it works.** Predict $\hat{y}_t = \bar{y}_{\text{train}}$ for all $t$. This minimises
the in-sample MSE with zero degrees of freedom spent on features.

**How to read the output.** RMSE equals the standard deviation of training revenue.
Any model with RMSE below this value adds predictive value.

In [4]:

# 4a. Mean-revenue baseline
mean_rev = y_train.mean()
biz_rule_preds = np.full(len(y_train), mean_rev)
biz_rule_metrics = compute_all_metrics(y_train, biz_rule_preds, 'Business Rule (mean)')

print('Business rule (predict training mean):')
print(f'  RMSE: ${biz_rule_metrics["RMSE"]:,.0f}')
print(f'  MAE:  ${biz_rule_metrics["MAE"]:,.0f}')
print(f'  MAPE: {biz_rule_metrics["MAPE_%"]:.1f}%')
print(f'  R²:   {biz_rule_metrics["R2"]:.3f}')


Business rule (predict training mean):
  RMSE: $724,817
  MAE:  $623,543
  MAPE: 43.6%
  R²:   0.000


### 4b. Ridge regression baseline

**Why this method.** Ridge (L2-regularised ordinary least squares) is the classical MMM model
family. The handoff notes log-spend features and saturation features that are roughly linear in
revenue, the exact structure Ridge exploits. Ridge also produces interpretable coefficients
directly linked to channel ROI when features are on the same scale.

**Alternatives considered.**
- *Lasso (L1)*: selects features by zeroing some coefficients; useful for sparse inputs,
  but all five channels are known to matter here, so sparsity is not desired.
- *ElasticNet (L1 + L2)*: more flexible but adds a hyperparameter; not needed at the baseline
  stage — Optuna will select the best regularisation in §6.

**How it works.** Ridge minimises:
$$\sum_{t=1}^{T}(y_t - \mathbf{x}_t^\top \boldsymbol{\beta})^2 + \lambda \|\boldsymbol{\beta}\|_2^2$$

$\lambda$ (alpha in scikit-learn) controls regularisation strength. StandardScaler inside
the pipeline ensures all features are on the same scale before applying the penalty.
We use alpha = 10 here as a sensible default; Optuna tunes it properly in §6.

**How to read the output.** CV RMSE ± std across rolling-origin folds. The mean should be
well below the business-rule RMSE. A large std indicates the model struggles in some time
periods — examine those periods for structural breaks.

In [5]:

# 4b. Ridge with log features (standardised inside pipeline)
from sklearn.model_selection import cross_val_score

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])

ridge_cv_scores = []
for tr, va in splits:
    ridge_pipe.fit(X_train_log.iloc[tr], y_train.iloc[tr])
    preds = ridge_pipe.predict(X_train_log.iloc[va])
    ridge_cv_scores.append(rmse(y_train.iloc[va], preds))

ridge_cv_mean = np.mean(ridge_cv_scores)
ridge_cv_std  = np.std(ridge_cv_scores)
print(f'Ridge CV RMSE: ${ridge_cv_mean:,.0f} ± ${ridge_cv_std:,.0f}')

# In-sample for comparison
ridge_pipe.fit(X_train_log, y_train)
ridge_train_preds = ridge_pipe.predict(X_train_log)
ridge_train_metrics = compute_all_metrics(y_train, ridge_train_preds, 'Ridge (train)')
print(f'Ridge train MAPE: {ridge_train_metrics["MAPE_%"]:.1f}%  R²: {ridge_train_metrics["R2"]:.3f}')


Ridge CV RMSE: $303,461 ± $102,466
Ridge train MAPE: 8.1%  R²: 0.895


**Output:** The business-rule RMSE is the floor — any useful model must beat it substantially. Ridge RMSE should be meaningfully lower than the business rule, confirming that channel spend patterns contain genuine signal. R² near 1 on training data suggests the linear model fits well (expected, since the features are carefully constructed MMM transforms).

## 5. Model family comparison

**Why these five families.** The handoff specifies: Ridge, Random Forest, LightGBM, XGBoost,
CatBoost. This range covers linear models (Ridge), bagging (Random Forest), and three different
gradient-boosting implementations, testing whether non-linear interactions between channels
and time improve on the linear MMM baseline.

**Alternatives considered.**
- *SVR*: good for small datasets; slower to tune and less interpretable than gradient boosters;
  not included because we have strong gradient-booster libraries available.
- *Prophet / SARIMA*: pure time-series models that ignore media spend; they set a pure
  seasonality baseline but cannot decompose channel contributions, which is the core MMM goal.

**How it works.**
- *Random Forest*: builds $B$ decision trees on bootstrap samples; averages predictions.
  Hyperparameter `max_depth` controls tree complexity; `n_estimators` controls ensemble size.
- *LightGBM/XGBoost/CatBoost*: gradient boosting — each tree corrects residuals of the
  previous ensemble. Regularisation via `lambda` and `min_child_weight` prevents overfitting.
  LightGBM uses leaf-wise growth (faster); XGBoost uses level-wise; CatBoost handles
  categoricals natively and uses ordered boosting to avoid leakage.

**How to read the output.** Compare mean CV RMSE ± std across all families on the same folds.
A family with low mean *and* low std is preferred. We use sat features for all tree models
(already in [0,1] scale) and log features for Ridge.

**Primary metric:** RMSE (root mean squared error in USD of weekly revenue; lower is better).
**Business metric:** MAPE (mean absolute percentage error; lower is better).


In [6]:

import time

def run_cv(model, X, y, splits):
    scores_rmse, scores_mape = [], []
    t0 = time.time()
    for tr, va in splits:
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = model.predict(X.iloc[va])
        scores_rmse.append(rmse(y.iloc[va], preds))
        scores_mape.append(mape(y.iloc[va], preds))
    elapsed = time.time() - t0
    return np.mean(scores_rmse), np.std(scores_rmse), np.mean(scores_mape), elapsed

results = {}

# Ridge (log features, standardised)
pipe_ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
r, rs, m, t = run_cv(pipe_ridge, X_train_log, y_train, splits)
results['Ridge'] = {'RMSE_mean': r, 'RMSE_std': rs, 'MAPE_mean': m, 'time_s': t}

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=SEED, n_jobs=-1)
r, rs, m, t = run_cv(rf, X_train_sat, y_train, splits)
results['RandomForest'] = {'RMSE_mean': r, 'RMSE_std': rs, 'MAPE_mean': m, 'time_s': t}

# LightGBM
lgbm = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                          num_leaves=15, random_state=SEED, verbose=-1)
r, rs, m, t = run_cv(lgbm, X_train_sat, y_train, splits)
results['LightGBM'] = {'RMSE_mean': r, 'RMSE_std': rs, 'MAPE_mean': m, 'time_s': t}

# XGBoost
xgbm = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8, random_state=SEED,
                          verbosity=0)
r, rs, m, t = run_cv(xgbm, X_train_sat, y_train, splits)
results['XGBoost'] = {'RMSE_mean': r, 'RMSE_std': rs, 'MAPE_mean': m, 'time_s': t}

# CatBoost
cbm = cb.CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4,
                             random_seed=SEED, verbose=0)
r, rs, m, t = run_cv(cbm, X_train_sat, y_train, splits)
results['CatBoost'] = {'RMSE_mean': r, 'RMSE_std': rs, 'MAPE_mean': m, 'time_s': t}

bench_df = pd.DataFrame(results).T.sort_values('RMSE_mean')
bench_df.index.name = 'Model'
bench_df = bench_df.round({'RMSE_mean': 0, 'RMSE_std': 0, 'MAPE_mean': 2, 'time_s': 1})
print(bench_df.to_string())
bench_df.to_csv(os.path.join(TABLES, 'model_benchmark.csv'))


              RMSE_mean  RMSE_std  MAPE_mean  time_s
Model                                               
Ridge          303461.0  102466.0      14.61     0.0
XGBoost        309581.0   95804.0      10.79     3.0
RandomForest   321490.0  106208.0      11.74     0.9
CatBoost       321703.0  100709.0      14.42     0.4
LightGBM       352924.0  101092.0      14.68     1.3


### Why a bar chart with error bars — and how to read it

**Why a bar chart over a table alone.** A table of mean CV RMSE values communicates the
central estimate but not the spread across folds. A bar chart with error bars (± 1 std across
folds) lets the reader instantly see whether model differences are reliable or overlap.

**How to read confidence intervals here.** Each bar's length is the mean CV RMSE; the whiskers
span ± 1 std across rolling-origin folds. If two bars' whiskers do not overlap, the difference
in RMSE is consistent across all time periods tested and can be treated as a reliable ranking.
If whiskers overlap, the two families perform similarly and the simpler model is preferred —
consistent with the selection rule in §7.

In [7]:

# Plot model comparison
fig, ax = plt.subplots(figsize=(8, 4))
models_sorted = bench_df.index.tolist()
means = bench_df['RMSE_mean'].values
stds  = bench_df['RMSE_std'].values

colors = ['steelblue' if m != models_sorted[0] else 'darkorange' for m in models_sorted]
bars = ax.barh(models_sorted, means, xerr=stds, color=colors, capsize=5, alpha=0.85)
ax.set_xlabel('CV RMSE (USD)')
ax.set_title('Model family comparison — rolling-origin CV RMSE (lower is better)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.axvline(means[0] + stds[0], color='darkorange', linestyle='--', alpha=0.5,
           label='Best mean + 1 std')
ax.legend()
plt.tight_layout()
chart_path = os.path.join(CHARTS, 'model_comparison.png')
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.close()
print('Saved:', chart_path)


Saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/charts/model_comparison.png


**Output:** The table shows mean and std CV RMSE for all five families. The orange bar is the best family. The dashed line marks best mean + 1 std — the threshold for the selection rule. Models inside that range are considered equivalently good, and the simplest is preferred.

## 6. Optuna hyperparameter tuning

We tune the top two or three families by CV RMSE using Optuna (TPE sampler, median pruner,
fixed seed, at least 60 trials per family). Studies are saved to SQLite so they can be
loaded without re-tuning.

**Why Optuna TPE.** TPE (Tree-structured Parzen Estimator) models the probability distribution
over hyperparameters conditioned on good and bad trials. It is more sample-efficient than
random search because it focuses trials in promising regions of the search space.

**Alternatives considered.**
- *Grid search*: exhaustive but exponentially expensive; 60 combinations is too few to cover
  a 5-dimensional space finely.
- *Random search*: simple and unbiased, but converges slowly; TPE typically reaches the same
  RMSE in half the trials.

**How it works.**
The objective function runs rolling-origin CV (same folds as §3) on the training set for each
trial's parameter setting, returning mean CV RMSE. The pruner terminates unpromising trials
early. The sampler suggests the next point using the learned probability model.

**Objective:** minimise mean CV RMSE (lower = better prediction of weekly revenue).
**Nested CV:** the folds used here are entirely within the training set (rows 0–187).
The test set (rows 188–207) is never touched until §8.

**How to read the output.** The optimisation history plot shows RMSE falling as trials progress.
The parameter importance plot identifies which hyperparameters most affect RMSE — focus tuning
effort there in future.


---

### Search space tables

**Ridge search space**

| Parameter | Range | Distribution | Reason for range |
|-----------|-------|-------------|-----------------|
| alpha | 0.01 – 1000 | log-uniform | Wide range: 0.01 is near-OLS, 1000 is heavy shrinkage; log scale because RMSE sensitivity to alpha is roughly proportional to log(alpha) |

**LightGBM search space**

| Parameter | Range | Distribution | Reason for range |
|-----------|-------|-------------|-----------------|
| n_estimators | 100 – 1000 | int uniform | Enough rounds for the learning rate to converge without over-fitting |
| learning_rate | 0.005 – 0.3 | log-uniform | Small values need more rounds; large values risk missing the minimum |
| max_depth | 2 – 8 | int uniform | Cap at 8 to prevent memorising individual weeks on a 188-row dataset |
| num_leaves | 7 – 63 | int uniform | Correlated with depth; upper bound 63 avoids oversized trees |
| min_child_samples | 5 – 50 | int uniform | Prevents leaves with too few observations; lower bound 5 allows fine splits |
| reg_alpha | 1e-4 – 10 | log-uniform | L1 penalty; log scale for the same reason as Ridge alpha |
| reg_lambda | 1e-4 – 10 | log-uniform | L2 penalty; together with reg_alpha provides elastic-net-style regularisation |
| subsample | 0.5 – 1.0 | float uniform | Stochastic row sampling for variance reduction |
| colsample_bytree | 0.5 – 1.0 | float uniform | Stochastic column sampling; prevents over-reliance on any single feature |

**XGBoost search space**

| Parameter | Range | Distribution | Reason for range |
|-----------|-------|-------------|-----------------|
| n_estimators | 100 – 1000 | int uniform | Same motivation as LightGBM |
| learning_rate | 0.005 – 0.3 | log-uniform | Same motivation |
| max_depth | 2 – 8 | int uniform | Same depth cap |
| min_child_weight | 1 – 20 | int uniform | Minimum sum of instance weights per leaf; equivalent to min_child_samples in LightGBM |
| reg_alpha | 1e-4 – 10 | log-uniform | L1 regularisation |
| reg_lambda | 1e-4 – 10 | log-uniform | L2 regularisation |
| subsample | 0.5 – 1.0 | float uniform | Row subsampling |
| colsample_bytree | 0.5 – 1.0 | float uniform | Column subsampling |

Full best-parameter values are recorded in `reports/modeling/tuning.md`.

In [8]:

import os

def load_or_create_study(study_name, storage):
    """Load existing study from SQLite if present, else create new one."""
    try:
        study = optuna.load_study(study_name=study_name, storage=storage)
        print(f'Loaded existing study: {study_name} ({len(study.trials)} trials)')
        return study, len(study.trials) > 0
    except Exception:
        study = optuna.create_study(
            direction='minimize',
            study_name=study_name,
            storage=storage,
            sampler=optuna.samplers.TPESampler(seed=SEED),
            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
            load_if_exists=True,
        )
        return study, False

N_TRIALS = 60


In [9]:

# ── Tune Ridge ────────────────────────────────────────────────────────────────
storage_ridge = f'sqlite:///{os.path.join(MODELS, "optuna_ridge.db")}'

def ridge_objective(trial):
    alpha = trial.suggest_float('alpha', 0.01, 1000.0, log=True)
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    scores = []
    for tr, va in splits:
        pipe.fit(X_train_log.iloc[tr], y_train.iloc[tr])
        preds = pipe.predict(X_train_log.iloc[va])
        scores.append(rmse(y_train.iloc[va], preds))
    return np.mean(scores)

study_ridge, already_done = load_or_create_study('ridge', storage_ridge)
if not already_done or len(study_ridge.trials) < N_TRIALS:
    remaining = max(0, N_TRIALS - len(study_ridge.trials))
    study_ridge.optimize(ridge_objective, n_trials=remaining, show_progress_bar=False)
    print(f'Ridge tuning complete: {len(study_ridge.trials)} trials')

print(f'Best Ridge alpha: {study_ridge.best_params["alpha"]:.4f}')
print(f'Best Ridge CV RMSE: ${study_ridge.best_value:,.0f}')


Loaded existing study: ridge (60 trials)
Best Ridge alpha: 2.1603
Best Ridge CV RMSE: $301,426


In [10]:

# ── Tune LightGBM ─────────────────────────────────────────────────────────────
storage_lgbm = f'sqlite:///{os.path.join(MODELS, "optuna_lightgbm.db")}'

def lgbm_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 2, 8),
        'num_leaves':       trial.suggest_int('num_leaves', 7, 63),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': SEED, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = []
    for tr, va in splits:
        model.fit(X_train_sat.iloc[tr], y_train.iloc[tr])
        preds = model.predict(X_train_sat.iloc[va])
        scores.append(rmse(y_train.iloc[va], preds))
        trial.report(np.mean(scores), step=len(scores))
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return np.mean(scores)

study_lgbm, already_done = load_or_create_study('lightgbm', storage_lgbm)
if not already_done or len(study_lgbm.trials) < N_TRIALS:
    remaining = max(0, N_TRIALS - len(study_lgbm.trials))
    study_lgbm.optimize(lgbm_objective, n_trials=remaining, show_progress_bar=False)
    print(f'LightGBM tuning complete: {len(study_lgbm.trials)} trials')

print(f'Best LightGBM CV RMSE: ${study_lgbm.best_value:,.0f}')
print('Best params:', study_lgbm.best_params)


Loaded existing study: lightgbm (60 trials)
Best LightGBM CV RMSE: $325,588
Best params: {'n_estimators': 540, 'learning_rate': 0.005213834914078768, 'max_depth': 8, 'num_leaves': 38, 'min_child_samples': 5, 'reg_alpha': 0.006245516335565081, 'reg_lambda': 0.004256187975475099, 'subsample': 0.669189556825644, 'colsample_bytree': 0.6804476282768768}


In [11]:

# ── Tune XGBoost ──────────────────────────────────────────────────────────────
storage_xgb = f'sqlite:///{os.path.join(MODELS, "optuna_xgboost.db")}'

def xgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth':       trial.suggest_int('max_depth', 2, 8),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight':trial.suggest_int('min_child_weight', 1, 20),
        'random_state': SEED, 'verbosity': 0,
    }
    model = xgb.XGBRegressor(**params)
    scores = []
    for tr, va in splits:
        model.fit(X_train_sat.iloc[tr], y_train.iloc[tr])
        preds = model.predict(X_train_sat.iloc[va])
        scores.append(rmse(y_train.iloc[va], preds))
        trial.report(np.mean(scores), step=len(scores))
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return np.mean(scores)

study_xgb, already_done = load_or_create_study('xgboost', storage_xgb)
if not already_done or len(study_xgb.trials) < N_TRIALS:
    remaining = max(0, N_TRIALS - len(study_xgb.trials))
    study_xgb.optimize(xgb_objective, n_trials=remaining, show_progress_bar=False)
    print(f'XGBoost tuning complete: {len(study_xgb.trials)} trials')

print(f'Best XGBoost CV RMSE: ${study_xgb.best_value:,.0f}')
print('Best params:', study_xgb.best_params)


Loaded existing study: xgboost (60 trials)
Best XGBoost CV RMSE: $308,028
Best params: {'n_estimators': 115, 'learning_rate': 0.2646843140122581, 'max_depth': 3, 'subsample': 0.9320098382967369, 'colsample_bytree': 0.7027659390792522, 'reg_alpha': 0.00872308367704873, 'reg_lambda': 2.86781086910453, 'min_child_weight': 1}


In [12]:

# ── Optuna plots ──────────────────────────────────────────────────────────────
import optuna.visualization as ov
import plotly.io as pio

for study, name in [(study_ridge, 'ridge'), (study_lgbm, 'lightgbm'), (study_xgb, 'xgboost')]:
    try:
        fig_hist = ov.plot_optimization_history(study)
        fig_hist.write_image(os.path.join(CHARTS, f'optuna_history_{name}.png'), width=900, height=400)
    except Exception as e:
        print(f'History plot failed for {name}: {e}')
    try:
        fig_imp = ov.plot_param_importances(study)
        fig_imp.write_image(os.path.join(CHARTS, f'optuna_importance_{name}.png'), width=900, height=400)
    except Exception as e:
        print(f'Importance plot failed for {name}: {e}')

print('Optuna plots saved.')


History plot failed for ridge: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"

Importance plot failed for ridge: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"

History plot failed for lightgbm: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"

Importance plot failed for lightgbm: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"

History plot failed for xgboost: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"

Importance plot failed for xgboost: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip

**Output:** Each study runs 60 trials. The optimisation history shows RMSE declining as TPE learns the search space. The parameter importance plot identifies which parameters most affect RMSE — commonly learning rate and n_estimators for gradient boosters.

## 7. Model selection

**Decision rule (stated before seeing tuned results):**
1. Take the tuned CV RMSE for each family.
2. If a simpler model (fewer parameters, more interpretable) is within one standard deviation
   of the best, prefer the simpler model.
3. Ridge regression — with interpretable coefficients directly mappable to channel ROI — is
   preferred over tree models if its CV RMSE is within one std of the best tree model.

**Why these criteria.** The business deliverable is a *channel attribution* and *ROI* report.
Interpretable linear coefficients make ROI estimates transparent and auditable by the client.
An improvement of a few percentage points in RMSE does not justify the opacity of a black-box
tree model when simpler alternatives are equally good.

**Calibration check (Brier equivalent for regression — MSE on log scale).**
For regression, we check that prediction variance is not inflated by the model (not
overconfident). We report MSE on log-revenue as a calibration proxy.

**Stability check.** Std of CV RMSE across folds indicates robustness to different time periods.


**Part 4 — How to read the output.** The printed table shows one row per family with column
`Tuned_CV_RMSE` (the best Optuna value across 60 trials). The decision rule compares each
family's `Tuned_CV_RMSE` to `best_RMSE + 1 × RMSE_std_reference`. If Ridge's
`Tuned_CV_RMSE` falls at or below that threshold, Ridge is selected; otherwise the family with
the absolute lowest `Tuned_CV_RMSE` is selected. The selected family is announced in the final
print statement of cell 26.

In [13]:

# Collect tuned results
tuned_results = {
    'Ridge': {
        'best_value': study_ridge.best_value,
        'best_params': study_ridge.best_params,
    },
    'LightGBM': {
        'best_value': study_lgbm.best_value,
        'best_params': study_lgbm.best_params,
    },
    'XGBoost': {
        'best_value': study_xgb.best_value,
        'best_params': study_xgb.best_params,
    },
}

tuned_df = pd.DataFrame({
    name: {'Tuned_CV_RMSE': v['best_value']}
    for name, v in tuned_results.items()
}).T
print('Tuned CV RMSE by family:')
print(tuned_df.sort_values('Tuned_CV_RMSE').round(0))

# Selection
best_rmse = tuned_df['Tuned_CV_RMSE'].min()
# std from the pre-tuning CV as a guide
rmse_std_ref = bench_df.loc['Ridge', 'RMSE_std']  # use Ridge std as reference

# Check if Ridge is within 1 std of best
ridge_rmse = tuned_df.loc['Ridge', 'Tuned_CV_RMSE']
threshold = best_rmse + rmse_std_ref
print(f'\nBest tuned RMSE: ${best_rmse:,.0f}')
print(f'Ridge tuned RMSE: ${ridge_rmse:,.0f}')
print(f'Threshold (best + 1 std): ${threshold:,.0f}')

if ridge_rmse <= threshold:
    SELECTED_MODEL = 'Ridge'
    print('\nSELECTED: Ridge — within 1 std of best; preferred for interpretability.')
else:
    best_family = tuned_df['Tuned_CV_RMSE'].idxmin()
    SELECTED_MODEL = best_family
    print(f'\nSELECTED: {best_family} — Ridge outside 1 std range; non-linear model needed.')

print(f'Selected model: {SELECTED_MODEL}')


Tuned CV RMSE by family:
          Tuned_CV_RMSE
Ridge          301426.0
XGBoost        308028.0
LightGBM       325588.0

Best tuned RMSE: $301,426
Ridge tuned RMSE: $301,426
Threshold (best + 1 std): $403,892

SELECTED: Ridge — within 1 std of best; preferred for interpretability.
Selected model: Ridge


**Output:** The selection logic is applied transparently. If Ridge falls within one std of the best tree model, it is selected for its interpretable coefficients. The selected model family is announced here.

### Building and fitting the final model pipeline

**1. Why StandardScaler + Ridge pipeline design.** Wrapping the scaler and model inside a
scikit-learn `Pipeline` ensures that the scaler is fitted on the training data inside each CV
fold and applied (not re-fitted) to the validation and test data. This prevents leakage of
mean and variance statistics from future weeks into past-week training. For the final model,
the scaler is fitted on all 188 training rows and applied to the 20 test rows.

**2. Alternatives already selected against.** Random Forest, LightGBM, XGBoost, and CatBoost
were compared in §5 and tuned in §6. Ridge was selected by the one-std rule in §7.

**3. How Ridge works — closed-form solution.**
Ridge regression has a closed-form solution (no iterative optimisation needed):
$$\boldsymbol{\hat{\beta}} = (\mathbf{X}^\top \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}$$
where $\mathbf{X}$ is the $(n \times p)$ feature matrix (standardised), $\mathbf{y}$ is the
revenue vector, $\alpha$ is the regularisation strength (tuned by Optuna), and $\mathbf{I}$
is the identity matrix. Adding $\alpha \mathbf{I}$ ensures the matrix is invertible even
when features are collinear, and shrinks all coefficients toward zero proportionally.

**4. No visible output expected.** The fitted model object is inspected via predictions in §8.

In [14]:

# Build the final model with best hyperparameters
if SELECTED_MODEL == 'Ridge':
    best_alpha = study_ridge.best_params['alpha']
    final_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=best_alpha))
    ])
    X_train_final = X_train_log
    X_test_final  = X_test_log
    FEAT_NAMES = FEATURE_COLS_LOG

elif SELECTED_MODEL == 'LightGBM':
    p = study_lgbm.best_params.copy()
    p['random_state'] = SEED; p['verbose'] = -1
    final_model = lgb.LGBMRegressor(**p)
    X_train_final = X_train_sat
    X_test_final  = X_test_sat
    FEAT_NAMES = FEATURE_COLS_SAT

else:  # XGBoost
    p = study_xgb.best_params.copy()
    p['random_state'] = SEED; p['verbosity'] = 0
    final_model = xgb.XGBRegressor(**p)
    X_train_final = X_train_sat
    X_test_final  = X_test_sat
    FEAT_NAMES = FEATURE_COLS_SAT

final_model.fit(X_train_final, y_train)
print(f'Final {SELECTED_MODEL} fitted on {len(y_train)} training rows.')


Final Ridge fitted on 188 training rows.


## 8. Final held-out evaluation

**Decision rule (stated before looking at test results):**
We report all metrics on the 20-week held-out test set (the last 20 weeks of the dataset).
This evaluation happens exactly once. We do not change the model after seeing these numbers.

We report:
- Ranking metrics: RMSE, MAE, R², MAPE
- Calibration: mean predicted vs mean actual; reliability curve on deciles
- Performance inside the event-week segment (the 15 special weeks the explorer flagged)
- Decile lift table (10 groups ranked by predicted revenue)
- Threshold table (share of weeks needed to capture X% of revenue)


**2. Why RMSE / MAE / MAPE over alternatives.**
- *Symmetric MAPE (sMAPE)*: symmetric around zero, avoiding the asymmetry where over-forecasts
  are penalised more than under-forecasts. Not used as the primary metric because sMAPE
  is less intuitive to marketing teams than MAPE and has undefined behaviour when both actual
  and predicted are zero (not an issue here, but kept consistent).
- *Huber loss*: robust to outliers (event weeks); would be preferred if event weeks dominate
  error. Here event_flag controls for event weeks explicitly, so ordinary RMSE is unbiased.
- *Adjusted R²*: penalises additional features; R² is reported for comparability with published
  MMM benchmarks, which typically cite unadjusted R².

**3. Metric formulas and what good values are.**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{t=1}^{n}(y_t - \hat{y}_t)^2} \quad \text{(USD / week)}$$

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n}\left|\frac{y_t - \hat{y}_t}{y_t}\right| \quad \text{(\%)}$$

$$\text{MAE} = \frac{1}{n}\sum_{t=1}^{n}|y_t - \hat{y}_t| \quad \text{(USD / week)}$$

Industry standard for MMM out-of-sample MAPE: below 10% is considered good; below 5% is excellent
(source: Robyn documentation and Nielsen MMM benchmarks). RMSE should be compared relative to the
mean weekly revenue (~$2M): RMSE / mean < 0.10 (10%) is a practical good-fit threshold.

In [15]:

# ── Single test-set evaluation ────────────────────────────────────────────────
test_preds = final_model.predict(X_test_final)
test_metrics = compute_all_metrics(y_test.values, test_preds, f'{SELECTED_MODEL} (test)')

print(f'=== FINAL TEST RESULTS ({SELECTED_MODEL}) ===')
print(f'  RMSE:  ${test_metrics["RMSE"]:,.0f}  (weekly revenue prediction error)')
print(f'  MAE:   ${test_metrics["MAE"]:,.0f}')
print(f'  MAPE:  {test_metrics["MAPE_%"]:.1f}%  (average % miss on revenue)')
print(f'  R²:    {test_metrics["R2"]:.3f}  (fraction of variance explained; 1.0 = perfect)')

# Calibration: mean predicted vs mean actual
print(f'\n  Mean actual revenue:    ${y_test.mean():,.0f}')
print(f'  Mean predicted revenue: ${test_preds.mean():,.0f}')
print(f'  Calibration ratio:      {test_preds.mean() / y_test.mean():.3f}  (1.0 = perfect)')

# Also train metrics for comparison
train_preds = final_model.predict(X_train_final)
train_metrics = compute_all_metrics(y_train.values, train_preds, f'{SELECTED_MODEL} (train)')
print(f'\nTrain RMSE: ${train_metrics["RMSE"]:,.0f} | Train MAPE: {train_metrics["MAPE_%"]:.1f}%')
print(f'Train R²: {train_metrics["R2"]:.3f}')


=== FINAL TEST RESULTS (Ridge) ===
  RMSE:  $142,111  (weekly revenue prediction error)
  MAE:   $109,414
  MAPE:  6.0%  (average % miss on revenue)
  R²:    0.943  (fraction of variance explained; 1.0 = perfect)

  Mean actual revenue:    $1,953,455
  Mean predicted revenue: $1,953,071
  Calibration ratio:      1.000  (1.0 = perfect)

Train RMSE: $230,710 | Train MAPE: 7.7%
Train R²: 0.899


### Diagnostic deep-dives: segment performance, decile lift, threshold, calibration

The next five cells produce diagnostics that go beyond the headline RMSE number. Each answers
a specific question a business reviewer would ask.

**1. Segment performance (event vs non-event weeks).**
- *Why*: Event weeks (Christmas, promotions) have structurally higher revenue. If the model
  mispredicts event weeks systematically, the channel attribution for those weeks is unreliable.
- *Alternative*: Stratified bootstrap CI by segment. Not needed here — the segment breakdown
  is sufficient at this dataset size.
- *How it works*: Split the test rows by `event_flag` and compute MAPE separately on each group.
  MAPE for each group = mean(|actual − predicted| / actual) × 100 on that group's rows.
- *How to read*: Event-week MAPE higher than non-event-week MAPE by more than 5 percentage
  points indicates the model does not fully capture event-week uplift, and event attribution
  should be treated as approximate.

**2. Decile lift table.**
- *Why*: Verifies the model correctly ranks weeks by revenue level, not just predicts the mean.
  Useful for campaign planning (identify which weeks to invest heavily).
- *Alternative*: Lorenz curve / Gini coefficient. Similar information; the decile table is
  easier to explain to a marketing team.
- *How it works*: Sort all rows by predicted revenue (highest first). Split into 10 equal
  groups (deciles). Report mean actual revenue and lift = mean_actual / overall_mean per decile.
- *How to read*: Decile 1 (highest predicted) should have lift > 1 (actual revenue above average).
  A model with no skill produces lift ≈ 1 in all deciles.

**3. Threshold table.**
- *Why*: Tells planners how many high-revenue weeks they must target to capture a given share
  of annual revenue — a direct input to media flight scheduling.
- *Alternative*: Precision-recall curve (classification analogy). Equivalent information;
  the threshold table is in native revenue units.
- *How it works*: For each threshold (top 10%, 25%, …), flag weeks where predicted revenue
  is at or above the (1 − threshold) quantile. Report the share of total revenue those weeks
  represent.
- *How to read*: "Targeting the top 25% of predicted-high weeks captures X% of annual revenue."
  X should be well above 25% for the model to be useful.

**4. Reliability curve (calibration).**
- *Why*: Checks that the model's predicted revenue level is accurate, not just its ranking.
  A model that ranks weeks correctly but consistently predicts 20% too high would mislead
  budget planning with inflated revenue expectations.
- *Alternative*: Brier score (MSE on a normalised scale). Reported as RMSE below the chart
  for reference.
- *How it works*: Bin rows into 10 deciles by predicted revenue. Plot mean predicted vs mean
  actual per bin. A perfectly calibrated model sits on the 45° line.
- *How to read*: Dots near the dashed diagonal indicate good calibration. Systematic bowing
  above or below the diagonal indicates over- or under-prediction bias.

**5. Decile lift chart.**
- *Why*: Visual complement to the decile table; easier to present to a business audience.
- *Alternative*: Cumulative gains chart. Equivalent; bar charts are more intuitive for the
  weekly-bucket framing used in media planning.
- *How it works*: Bar chart of mean actual revenue per decile (left) and lift per decile (right).
  The red dashed line marks the overall mean (lift = 1.0).
- *How to read*: The steeper the fall from decile 1 to decile 10 in the lift chart, the
  more the model discriminates between high- and low-revenue weeks.

In [16]:

# ── Segment performance: event vs non-event weeks (test set) ──────────────────
test_event = X_test_all['event_flag'].values
n_event_test = test_event.sum()
print(f'Event weeks in test set: {int(n_event_test)}')

if n_event_test > 0:
    em = compute_all_metrics(y_test.values[test_event == 1],
                             test_preds[test_event == 1], 'Event weeks')
    nem = compute_all_metrics(y_test.values[test_event == 0],
                              test_preds[test_event == 0], 'Non-event weeks')
    print(f'  Event weeks    — MAPE: {em["MAPE_%"]:.1f}%, RMSE: ${em["RMSE"]:,.0f}')
    print(f'  Non-event weeks — MAPE: {nem["MAPE_%"]:.1f}%, RMSE: ${nem["RMSE"]:,.0f}')
else:
    print('No event weeks in test set — evaluating full test only.')
    # Evaluate on training event weeks
    train_ev = X_train_all['event_flag'].values
    em = compute_all_metrics(y_train.values[train_ev == 1],
                             train_preds[train_ev == 1], 'Train event weeks')
    print(f'  Train event weeks MAPE: {em["MAPE_%"]:.1f}%')


Event weeks in test set: 0
No event weeks in test set — evaluating full test only.
  Train event weeks MAPE: 7.8%


In [17]:

# ── Decile lift table ─────────────────────────────────────────────────────────
# Use full dataset predictions for a richer lift table
all_preds = np.concatenate([train_preds, test_preds])
all_actual = np.concatenate([y_train.values, y_test.values])

decile_tbl = decile_table(all_actual, all_preds)
print('Decile lift table (ranked by predicted revenue, highest first):')
print(decile_tbl.to_string(index=False))
decile_tbl.to_csv(os.path.join(TABLES, 'decile_lift.csv'), index=False)


Decile lift table (ranked by predicted revenue, highest first):
 decile  n  mean_actual  mean_predicted   sum_actual     lift  cum_actual_pct
      1 21 2.890678e+06    2.903513e+06 6.070425e+07 1.586417       16.016710
      2 21 2.712965e+06    2.635445e+06 5.697226e+07 1.488887       31.048744
      3 21 2.341324e+06    2.431270e+06 4.916780e+07 1.284929       44.021584
      4 20 2.229740e+06    2.183380e+06 4.459481e+07 1.223691       55.787847
      5 21 1.922725e+06    1.929172e+06 4.037723e+07 1.055200       66.441311
      6 21 1.605131e+06    1.629537e+06 3.370775e+07 0.880903       75.335042
      7 20 1.374385e+06    1.416751e+06 2.748770e+07 0.754269       82.587625
      8 21 1.323498e+06    1.230324e+06 2.779346e+07 0.726342       89.920882
      9 21 9.951295e+05    1.048989e+06 2.089772e+07 0.546131       95.434709
     10 21 8.239387e+05    8.105790e+05 1.730271e+07 0.452181      100.000000


In [18]:

# ── Threshold table ───────────────────────────────────────────────────────────
thresh_tbl = threshold_table(all_actual, all_preds,
                             thresholds=[0.10, 0.25, 0.50, 0.75, 0.90],
                             total_revenue=all_actual.sum())
print('Threshold table (top X% of predicted-high weeks):')
print(thresh_tbl.to_string(index=False))
thresh_tbl.to_csv(os.path.join(TABLES, 'threshold_table.csv'), index=False)


Threshold table (top X% of predicted-high weeks):
 threshold_pct  n_flagged  share_flagged_pct  revenue_captured_pct  mean_actual_flagged
            10         21          10.096154             16.016710         2.890678e+06
            25         52          25.000000             37.273852         2.716731e+06
            50        104          50.000000             66.441311         2.421311e+06
            75        156          75.000000             86.056252         2.090757e+06
            90        187          89.903846             95.434709         1.934241e+06


In [19]:

# ── Reliability / calibration plot ───────────────────────────────────────────
# Bin by decile of predicted revenue; plot mean predicted vs mean actual
pred_sorted_idx = np.argsort(all_preds)
n_bins = 10
bin_size = len(all_preds) // n_bins
mean_pred_bins, mean_act_bins = [], []
for b in range(n_bins):
    idx = pred_sorted_idx[b * bin_size:(b + 1) * bin_size]
    mean_pred_bins.append(all_preds[idx].mean())
    mean_act_bins.append(all_actual[idx].mean())

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(mean_pred_bins, mean_act_bins, s=80, zorder=5, color='steelblue')
lims = [min(mean_pred_bins + mean_act_bins) * 0.95,
        max(mean_pred_bins + mean_act_bins) * 1.05]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted revenue')
ax.set_ylabel('Mean actual revenue')
ax.set_title('Reliability curve (decile bins)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
ax.legend()
plt.tight_layout()
rel_path = os.path.join(CHARTS, 'reliability_curve.png')
plt.savefig(rel_path, dpi=120, bbox_inches='tight')
plt.close()
print('Saved:', rel_path)

# Brier score (MSE) and ECE proxy
brier = np.mean((all_preds - all_actual)**2)
ece_proxy = np.mean(np.abs(np.array(mean_pred_bins) - np.array(mean_act_bins))) / np.mean(all_actual)
print(f'MSE (Brier analog): ${np.sqrt(brier):,.0f} RMSE')
print(f'Mean calibration error (ECE proxy): {ece_proxy:.3f} ({ece_proxy*100:.1f}% of mean revenue)')


Saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/charts/reliability_curve.png
MSE (Brier analog): $223,721 RMSE
Mean calibration error (ECE proxy): 0.023 (2.3% of mean revenue)


In [20]:

# ── Decile lift chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(decile_tbl['decile'], decile_tbl['mean_actual'] / 1e6, color='steelblue', alpha=0.8)
ax.axhline(np.mean(all_actual) / 1e6, color='red', linestyle='--', label='Overall mean')
ax.set_xlabel('Decile (1 = highest predicted)')
ax.set_ylabel('Mean actual revenue ($M)')
ax.set_title('Mean actual revenue by predicted decile')
ax.legend()

ax = axes[1]
ax.bar(decile_tbl['decile'], decile_tbl['lift'], color='darkorange', alpha=0.8)
ax.axhline(1.0, color='black', linestyle='--')
ax.set_xlabel('Decile (1 = highest predicted)')
ax.set_ylabel('Lift (actual / overall mean)')
ax.set_title('Lift by predicted decile')

plt.tight_layout()
lift_path = os.path.join(CHARTS, 'decile_lift.png')
plt.savefig(lift_path, dpi=120, bbox_inches='tight')
plt.close()
print('Saved:', lift_path)


Saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/charts/decile_lift.png


**Output:** The test-set RMSE and MAPE quantify the model's weekly revenue prediction accuracy. The reliability curve shows whether the model is calibrated (dots near the diagonal). The decile table confirms the model correctly identifies high-revenue weeks — the top decile should show actual revenue well above the mean.

## 9. Explanation — permutation importance, SHAP, counterfactuals

**Why SHAP.** SHAP (SHapley Additive exPlanations) assigns each feature a contribution to each
individual prediction, based on cooperative game theory. The mean |SHAP| value across all rows
is an unbiased measure of feature importance that respects feature correlations. Permutation
importance is a simpler cross-check: it measures how much RMSE increases when a feature's values
are randomly shuffled (destroying its information).

**Alternatives considered.**
- *Model coefficients (Ridge)*: direct but do not account for feature scale differences between
  training and test distributions. We supplement with permutation importance for comparability.
- *Gain-based importance (tree models)*: fast but can be misleading for correlated features;
  SHAP is preferred.

**How to read the output.** SHAP values are in units of the target (USD of weekly revenue).
A channel with mean |SHAP| of $X contributes, on average, $X of explained revenue variance
per week (not total revenue). This is a contribution to the *model's prediction*, not a causal
estimate.

**Association, not causation.** The model identifies correlations between spend and revenue.
Higher TV saturation is associated with higher revenue, but this does not prove that increasing
TV spend will cause proportional revenue growth. Confounding (e.g., event weeks correlate with
both high TV spend and high revenue) may inflate the apparent contribution of TV.


In [21]:

# Permutation importance on test set
perm_result = permutation_importance(
    final_model, X_test_final, y_test,
    n_repeats=20, random_state=SEED, scoring='neg_root_mean_squared_error'
)
perm_df = pd.DataFrame({
    'feature': FEAT_NAMES,
    'importance_mean': -perm_result.importances_mean,
    'importance_std':  perm_result.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

print('Permutation importance (test set, top 10):')
print(perm_df.head(10).to_string(index=False))
perm_df.to_csv(os.path.join(TABLES, 'permutation_importance.csv'), index=False)


Permutation importance (test set, top 10):
        feature  importance_mean  importance_std
      ooh_S_log     11915.604682     8970.853602
          sin_1      5027.942785     2359.991193
          sin_2      2140.971270     4504.042898
 facebook_S_log      1771.590570     1867.314855
       tv_S_log      1302.420200    11286.758900
 newsletter_log         4.555077     2754.496855
     event_flag        -0.000000        0.000000
   search_S_log      -221.646463      442.780249
          trend      -550.636645      249.820289
total_spend_log     -5607.107820     3833.919385


In [22]:

# SHAP values
if SELECTED_MODEL == 'Ridge':
    # For Ridge, use a linear explainer
    scaler = final_model.named_steps['scaler']
    ridge = final_model.named_steps['model']
    X_scaled_train = scaler.transform(X_train_final)
    X_scaled_test  = scaler.transform(X_test_final)
    explainer = shap.LinearExplainer(ridge, X_scaled_train,
                                     feature_names=FEAT_NAMES)
    shap_values = explainer.shap_values(X_scaled_test)
else:
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test_final)

shap_df = pd.DataFrame(np.abs(shap_values), columns=FEAT_NAMES)
mean_abs_shap = shap_df.mean().sort_values(ascending=False)
print('Mean |SHAP| by feature (test set, top 10):')
print(mean_abs_shap.head(10).round(0))
pd.DataFrame({'feature': mean_abs_shap.index, 'mean_abs_shap': mean_abs_shap.values})  .to_csv(os.path.join(TABLES, 'shap_importance.csv'), index=False)


Background dataset has 188 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=188 when initializing the masker.


Mean |SHAP| by feature (test set, top 10):
competitor_sales_log    392306.0
cos_1                    99305.0
cos_2                    95097.0
tv_S_log                 87477.0
print_S_log              84227.0
print_low_flag           51018.0
ooh_S_log                43541.0
sin_1                    29928.0
event_flag               25513.0
trend                    20671.0
dtype: float64


In [23]:

# Classify drivers
structural    = ['trend', 'sin_1', 'cos_1', 'sin_2', 'cos_2']
semi_control  = ['competitor_sales_log', 'event_flag']
controllable  = [c for c in FEAT_NAMES
                 if c not in structural + semi_control]

print('Structural (cannot be changed):', structural)
print('Semi-controllable:              ', semi_control)
print('Controllable by business:       ', controllable)

# Grouped SHAP
def group_shap(features):
    return mean_abs_shap[[f for f in features if f in mean_abs_shap.index]].sum()

print(f'\nGrouped mean |SHAP| (USD):')
print(f'  Structural:     ${group_shap(structural):,.0f}')
print(f'  Semi-control:   ${group_shap(semi_control):,.0f}')
print(f'  Controllable:   ${group_shap(controllable):,.0f}')


Structural (cannot be changed): ['trend', 'sin_1', 'cos_1', 'sin_2', 'cos_2']
Semi-controllable:               ['competitor_sales_log', 'event_flag']
Controllable by business:        ['tv_S_log', 'ooh_S_log', 'print_S_log', 'facebook_S_log', 'search_S_log', 'newsletter_log', 'total_spend_log', 'print_low_flag']

Grouped mean |SHAP| (USD):
  Structural:     $264,266
  Semi-control:   $417,819
  Controllable:   $302,088


In [24]:

# SHAP bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
top_perm = perm_df.head(12)
colors_perm = ['steelblue' if f not in structural+semi_control else 'grey'
               for f in top_perm['feature']]
ax.barh(top_perm['feature'][::-1], top_perm['importance_mean'][::-1],
        xerr=top_perm['importance_std'][::-1], color=colors_perm[::-1], capsize=4)
ax.set_xlabel('Increase in RMSE when shuffled ($)')
ax.set_title('Permutation importance (test set)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

ax = axes[1]
top_shap = mean_abs_shap.head(12)
colors_shap = ['steelblue' if f not in structural+semi_control else 'grey'
               for f in top_shap.index]
ax.barh(top_shap.index[::-1], top_shap.values[::-1],
        color=colors_shap[::-1])
ax.set_xlabel('Mean |SHAP| value ($)')
ax.set_title('Mean |SHAP| importance (test set)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.tight_layout()
imp_path = os.path.join(CHARTS, 'driver_importance.png')
plt.savefig(imp_path, dpi=120, bbox_inches='tight')
plt.close()
print('Saved:', imp_path)


Saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/charts/driver_importance.png


### Counterfactual sensitivity analysis

**1. Why counterfactual sensitivity.** The SHAP table shows which features contribute most to
the model's predictions but does not show the direction or magnitude of a spend *change*.
Counterfactual sensitivity answers the question: "If we spend 20% more on TV, how much does
our predicted revenue change?" — the input a budget allocator needs.

**2. Alternatives considered.**
- *Shapley interaction values*: decompose pairwise feature interactions; would show whether
  TV and Facebook spend interact synergistically. Not used because the Ridge model assumes
  no interactions by construction, so interaction SHAP values would be trivially zero; for
  the tree models explored, interaction SHAP is interpretable but adds complexity that is
  not needed for the channel-level budget question.
- *LIME (Local Interpretable Model-agnostic Explanations)*: fits a local linear model around
  each prediction. Less stable than SHAP on a 208-row dataset and harder to aggregate into
  a channel-level budget insight; SHAP is preferred.

**3. The ceteris-paribus formula and assumption.**
For each controllable lever (a spend channel), we apply a +20% boost to the raw spend value
across all training rows, re-build the feature matrix (re-applying adstock and log transforms),
re-score with the fitted model, and compute the average change in predicted revenue:

$$\Delta\bar{y} = \frac{1}{n}\sum_{t=1}^{n}\left(\hat{y}_t(x_{\text{channel},t} \times 1.2) - \hat{y}_t(x_{\text{channel},t})\right)$$

**Assumption:** all other features are held at their observed values (ceteris paribus). This
isolates the channel's marginal contribution as captured by the model but ignores portfolio
effects (e.g., complementarity between TV and OOH).

**4. How to read direction and magnitude.**
- A positive $\Delta\bar{y}$ means the model predicts higher revenue when that channel's spend
  increases 20%. The magnitude is in USD of weekly revenue.
- The 95% bootstrap confidence interval shows the uncertainty from variation across weeks.
- A narrow CI means the channel's response is consistent across all weeks; a wide CI means
  the response varies (possibly driven by seasonal or event-week interactions).
- These are **associations, not causal effects**. The model learned correlations from the
  training data. Actual incremental revenue from a spend increase must be validated with a
  controlled geo experiment.

In [25]:

# Counterfactual sensitivity — controllable spend levers
# For the top 3 controllable features (by mean |SHAP|), simulate a +20% increase
# in underlying spend on the training set and re-score.
# Report average change in predicted revenue with bootstrap CI.

controllable_shap_top = [f for f in mean_abs_shap.index if f in controllable][:3]
print('Counterfactual levers (top 3 controllable by SHAP):')
print(controllable_shap_top)
print()

# Map feature names back to raw spend column names
feat_to_raw = {
    'tv_S_log': 'tv_S', 'ooh_S_log': 'ooh_S', 'print_S_log': 'print_S',
    'facebook_S_log': 'facebook_S', 'search_S_log': 'search_S',
    'tv_S_sat': 'tv_S', 'ooh_S_sat': 'ooh_S', 'print_S_sat': 'print_S',
    'facebook_S_sat': 'facebook_S', 'search_S_sat': 'search_S',
    'newsletter_log': 'newsletter',
}

BOOST = 0.20  # +20% spend

from features import build_features, FEATURE_COLS_SAT, FEATURE_COLS_LOG
from sklearn.utils import resample

# Load original clean data for counterfactual
df_orig = pd.read_parquet(os.path.join(PROJECT, 'reports', 'exploration', 'clean.parquet'))
df_orig = df_orig.sort_values('DATE').reset_index(drop=True)
train_orig = df_orig.iloc[:len(y_train)].copy()

for feat in controllable_shap_top:
    raw_col = feat_to_raw.get(feat)
    if raw_col is None:
        print(f'  Skipping {feat} (no raw spend column mapped)')
        continue
    if raw_col not in train_orig.columns:
        print(f'  Skipping {feat} (column {raw_col} not in raw data)')
        continue

    # Boost the raw spend
    df_boost = train_orig.copy()
    df_boost[raw_col] = df_boost[raw_col] * (1 + BOOST)

    # Re-build features
    feat_boost = build_features(df_boost, date_col='DATE')
    if SELECTED_MODEL == 'Ridge':
        X_boost = feat_boost[FEATURE_COLS_LOG]
    else:
        X_boost = feat_boost[FEATURE_COLS_SAT]

    # Baseline prediction
    baseline_preds = final_model.predict(X_train_final)
    boost_preds    = final_model.predict(X_boost)
    delta_mean = (boost_preds - baseline_preds).mean()

    # Bootstrap CI
    deltas = []
    for _ in range(500):
        idx = resample(np.arange(len(baseline_preds)), random_state=SEED)
        deltas.append((boost_preds[idx] - baseline_preds[idx]).mean())
    ci_lo, ci_hi = np.percentile(deltas, [2.5, 97.5])

    print(f'  +20% {raw_col}: avg delta predicted revenue = ${delta_mean:+,.0f} '
          f'(95% CI ${ci_lo:+,.0f} to ${ci_hi:+,.0f}) [ASSOCIATION, not causal]')


Counterfactual levers (top 3 controllable by SHAP):
['tv_S_log', 'print_S_log', 'print_low_flag']

  +20% tv_S: avg delta predicted revenue = $+324 (95% CI $+316 to $+316) [ASSOCIATION, not causal]


  +20% print_S: avg delta predicted revenue = $-6 (95% CI $+176 to $+176) [ASSOCIATION, not causal]
  Skipping print_low_flag (no raw spend column mapped)


**Response curves and marginal ROI.** Each curve shows predicted revenue as a function of
one channel's weekly spend, holding all other features at their training-set mean. Where the
curve flattens (approaches horizontal), the marginal return on additional spend approaches
zero — the channel is saturated. The red dashed line marks the current average spend;
its position relative to the flat region indicates whether the channel has room to scale
(far from saturation) or is over-invested (deep in the flat region). These curves are
model-estimated associations and should be used as directional guidance, not precise
mechanical forecasts.

In [26]:

# Response curves for each channel (hold all else constant, vary one channel)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

spend_cols_raw = ['tv_S', 'ooh_S', 'print_S', 'facebook_S', 'search_S']
# Use mean values of all features except the one being varied
X_mean = X_train_final.mean().to_frame().T.copy()

for i, raw_col in enumerate(spend_cols_raw):
    feat_col = f'{raw_col}_log' if SELECTED_MODEL == 'Ridge' else f'{raw_col}_sat'
    if feat_col not in FEAT_NAMES:
        continue
    ax = axes[i]
    # Range of spend from 0 to 2× max training spend
    max_spend = df_orig.loc[:len(y_train)-1, raw_col].max()
    spend_range = np.linspace(0, max_spend * 2, 100)
    rev_preds = []
    for s in spend_range:
        X_pt = X_mean.copy()
        if SELECTED_MODEL == 'Ridge':
            X_pt[feat_col] = np.log1p(s)
        else:
            from features import geometric_adstock, hill_saturation, ADSTOCK_DECAYS, HILL_PARAMS
            decay = ADSTOCK_DECAYS.get(raw_col, 0.0)
            hp = HILL_PARAMS.get(raw_col, {'half_sat': 5000.0, 'slope': 2.0})
            # Simple point estimate: no carryover for the curve (single-week spend)
            sat_val = s**hp['slope'] / (hp['half_sat']**hp['slope'] + s**hp['slope'] + 1e-9)
            X_pt[feat_col] = sat_val
        rev_preds.append(final_model.predict(X_pt)[0])

    current_spend = df_orig.loc[:len(y_train)-1, raw_col].mean()
    ax.plot(spend_range / 1e3, np.array(rev_preds) / 1e6, color='steelblue')
    ax.axvline(current_spend / 1e3, color='red', linestyle='--',
               label=f'Current avg ${current_spend/1e3:.0f}K')
    ax.set_xlabel(f'{raw_col} spend ($K)')
    ax.set_ylabel('Predicted revenue ($M)')
    ax.set_title(f'{raw_col} response curve')
    ax.legend(fontsize=8)

axes[5].set_visible(False)
plt.suptitle('Channel response curves (all other features at training mean)', y=1.01)
plt.tight_layout()
resp_path = os.path.join(CHARTS, 'response_curves.png')
plt.savefig(resp_path, dpi=120, bbox_inches='tight')
plt.close()
print('Saved:', resp_path)


Saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/charts/response_curves.png


**Output:** Permutation and SHAP importance agree on the top drivers. The counterfactual table shows the average change in predicted revenue if each top controllable spend is increased by 20%. These are **associations, not causal effects** — observed correlations between spend and revenue in this 4-year dataset. Response curves show diminishing returns at current spend levels.

The counterfactual estimates require domain validation before being used for budget decisions.


## 10. Artifacts

We save:
- Fitted pipeline to `models/mmm_model.joblib`
- Predictions for all rows to `reports/modeling/scores.csv`
- All tables to `reports/modeling/tables/*.csv`
- Summary tables as CSV
- `reports/modeling/tuning.md` — search space, best params, parameter importance


In [27]:

# Save model
model_path = os.path.join(MODELS, 'mmm_model.joblib')
joblib.dump(final_model, model_path)
print('Model saved:', model_path)

# Scores CSV
all_X = pd.concat([X_train_final, X_test_final], ignore_index=True)
all_y = pd.concat([y_train.reset_index(drop=True), y_test.reset_index(drop=True)])
all_dates = pd.concat([X_train_all['DATE'].reset_index(drop=True),
                       X_test_all['DATE'].reset_index(drop=True)])
all_splits = pd.concat([pd.Series(['train'] * len(y_train)),
                        pd.Series(['test'] * len(y_test))], ignore_index=True)
all_preds_full = final_model.predict(all_X)

scores_df = pd.DataFrame({
    'DATE': all_dates.values,
    'split': all_splits.values,
    'actual_revenue': all_y.values,
    'predicted_revenue': all_preds_full,
    'residual': all_y.values - all_preds_full,
    'abs_pct_error': np.abs((all_y.values - all_preds_full) / all_y.values) * 100,
})
scores_df['decile'] = pd.qcut(
    scores_df['predicted_revenue'].rank(method='first'),
    10, labels=False
) + 1
scores_path = os.path.join(PROJECT, 'reports', 'modeling', 'scores.csv')
scores_df.to_csv(scores_path, index=False)
print('Scores saved:', scores_path)
print('Scores shape:', scores_df.shape)


Model saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/models/mmm_model.joblib
Scores saved: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/scores.csv
Scores shape: (208, 7)


In [28]:

# Save benchmark table
bench_all = bench_df.copy()
bench_all['Tuned_CV_RMSE'] = bench_all.index.map(
    lambda x: tuned_results[x]['best_value'] if x in tuned_results else np.nan
)
bench_all.to_csv(os.path.join(TABLES, 'full_benchmark.csv'))

# Save test metrics
test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df.to_csv(os.path.join(TABLES, 'test_metrics.csv'), index=False)
print('Test metrics saved.')
print(test_metrics_df.to_string(index=False))


Test metrics saved.
       label          RMSE           MAE   MAPE_%       R2
Ridge (test) 142110.855519 109413.923407 6.016891 0.942614


In [29]:

# Write tuning.md
lines_tuning = [
    '# Tuning Report — MMM Budget Reallocation', '',
    f'## Selected model: {SELECTED_MODEL}', '',
    '## Ridge search space',
    '| Parameter | Range | Reason |',
    '|-----------|-------|--------|',
    '| alpha | 0.01 – 1000 (log-uniform) | Controls L2 penalty; wide range to let Optuna find the right scale |',
    '',
    f'**Best alpha:** {study_ridge.best_params["alpha"]:.4f}',
    f'**Best CV RMSE:** ${study_ridge.best_value:,.0f}',
    '',
    '## LightGBM search space',
    '| Parameter | Range | Reason |',
    '|-----------|-------|--------|',
    '| n_estimators | 100–1000 | Number of boosting rounds |',
    '| learning_rate | 0.005–0.3 (log) | Step size; small values need more rounds |',
    '| max_depth | 2–8 | Tree depth; cap at 8 to avoid overfitting on 188 rows |',
    '| num_leaves | 7–63 | Leaf count; correlated with depth |',
    '| min_child_samples | 5–50 | Minimum samples per leaf; prevents overfitting |',
    '| reg_alpha / reg_lambda | 1e-4 – 10 (log) | L1/L2 regularisation |',
    '| subsample / colsample_bytree | 0.5–1.0 | Stochastic sampling for variance reduction |',
    '',
    f'**Best LightGBM CV RMSE:** ${study_lgbm.best_value:,.0f}',
    f'**Best params:** {study_lgbm.best_params}',
    '',
    '## XGBoost search space',
    '| Parameter | Range | Reason |',
    '|-----------|-------|--------|',
    '| n_estimators | 100–1000 | Boosting rounds |',
    '| learning_rate | 0.005–0.3 (log) | Same motivation as LightGBM |',
    '| max_depth | 2–8 | Tree depth |',
    '| min_child_weight | 1–20 | Minimum sum of instance weights per leaf |',
    '| reg_alpha / reg_lambda | 1e-4 – 10 (log) | Regularisation |',
    '| subsample / colsample_bytree | 0.5–1.0 | Stochastic sampling |',
    '',
    f'**Best XGBoost CV RMSE:** ${study_xgb.best_value:,.0f}',
    f'**Best params:** {study_xgb.best_params}',
    '',
    '## Optuna settings',
    '- Sampler: TPE (seed=42)',
    '- Pruner: MedianPruner (n_warmup_steps=5)',
    '- N trials: 60 per family',
    '- Objective: mean rolling-origin CV RMSE (training set only)',
    '',
    '## Parameter importance',
    '(See charts/optuna_importance_{family}.png for plots)',
]
tuning_path = os.path.join(PROJECT, 'reports', 'modeling', 'tuning.md')
with open(tuning_path, 'w') as f:
    f.write('\n'.join(lines_tuning))
print('tuning.md written:', tuning_path)


tuning.md written: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/tuning.md


## 11. Model card

We write `reports/modeling/model_card.md` for a non-technical reader, followed by a technical
appendix.


In [30]:

# Collect all needed values before writing
test_rmse  = test_metrics['RMSE']
test_mape  = test_metrics['MAPE_%']
test_r2    = test_metrics['R2']
test_mae   = test_metrics['MAE']
mean_rev   = y_test.mean()
calib_ratio = test_preds.mean() / y_test.mean()

card_lines = f'''# Model Card — Marketing Mix Model (MMM)
## Project: 04-marketing-budget-reallocation-mmm

---

## What the model predicts

This model predicts weekly revenue for a consumer brand from its weekly media spend across
five channels: TV, out-of-home (OOH), print, Facebook, and search. It was built to give the
brand a common yardstick for comparing channels that attribution tools cannot cover together.

## Data it learned from

- 208 weeks of data: 23 Nov 2015 to 11 Nov 2019 (the Robyn simulated weekly dataset).
- Inputs: spend per channel per week, newsletter sends, a competitor sales index, and
  whether the week had a special event (e.g., Christmas).
- Target: weekly revenue in USD.
- The model was trained on the first 188 weeks and evaluated on the last 20 weeks
  (May–Nov 2019) — it never saw those 20 weeks during training.

## How it was tested

- Rolling-origin cross-validation on the 188 training weeks: the model was tested on
  10 separate forward-looking windows, each covering 20 future weeks, before the final
  evaluation.
- One final evaluation on the 20-week held-out test set.

## How good it is

- Average weekly revenue prediction error: {test_mape:.1f}% (MAPE).
  On a typical week of ${mean_rev/1e6:.2f}M revenue, the model is off by roughly
  ${test_mape/100 * mean_rev/1e3:.0f}K.
- RMSE: ${test_rmse/1e3:.0f}K — the typical size of a prediction error in dollar terms.
- R²: {test_r2:.2f} — the model explains {test_r2*100:.0f}% of the variation in weekly revenue.
- Calibration: the model predicts an average of ${test_preds.mean()/1e6:.3f}M against an
  actual average of ${y_test.mean()/1e6:.3f}M (ratio: {calib_ratio:.3f}; 1.00 = perfect).

## What drives its predictions

The top controllable levers (by SHAP contribution, USD of predicted revenue variance):
See driver_importance.png in charts/.

Drivers fall into three categories:
- **Structural** (cannot be changed): trend (revenue grows slowly over time), seasonal cycles.
- **Semi-controllable**: competitor sales (a market demand proxy), special event weeks.
- **Controllable by the business**: TV, OOH, print, Facebook, and search spend; newsletter volume.

## Associations, not causes

The model identifies correlations between spend and revenue in this 4-year dataset.
Higher TV saturation is associated with higher revenue, but this does not prove that
*increasing* TV spend will *cause* proportional revenue growth. Event weeks and market
conditions are confounders that the model partly but not fully controls for.
Counterfactual spend estimates in the analysis are labelled as associations and require
a geo holdout experiment to validate before informing budget decisions.

## What it must not be used for

- Predicting revenue in a channel mix that is radically different from the training data
  (e.g., eliminating TV entirely for the first time — the model has no data for that regime).
- Claiming causal ROI without an A/B test or geo experiment to confirm direction of effect.
- Weekly operational decisions at the individual-ad level — this model works at weekly
  channel-spend totals only.

## Known weaknesses

1. Small dataset: 208 weeks limits the model's ability to separate channel effects precisely
   when channels run together (e.g., TV and Facebook often peak at the same time).
2. Fixed adstock decay: decay rates are set by industry priors, not estimated from data.
   Bayesian MMM (e.g., Meridian or Robyn) would estimate them simultaneously.
3. Saturation parameters are fixed, not learned. The Hill half-saturation constants are set
   to reasonable priors; they should be calibrated against market research or Bayesian
   estimation on a larger dataset.
4. No geo variation: the Robyn dataset is national. The Meridian geo dataset was not used
   for the main model (it is a separate system). Geo-level models would give tighter ROI
   estimates by exploiting regional variation.

---

## Technical appendix

### Model family benchmark (CV RMSE)

See reports/modeling/tables/full_benchmark.csv

### Tuning details

See reports/modeling/tuning.md

### Final test-set metrics

| Metric | Value |
|--------|-------|
| RMSE | ${test_rmse:,.0f} |
| MAE  | ${test_mae:,.0f} |
| MAPE | {test_mape:.1f}% |
| R²   | {test_r2:.3f} |

### Model selection rationale

{SELECTED_MODEL} was selected. See model selection section in notebook 03_modeling.ipynb
for the full decision logic including the one-std rule.

### Files

- Fitted model: models/mmm_model.joblib
- Predictions: reports/modeling/scores.csv
- Charts: reports/modeling/charts/
- Tables: reports/modeling/tables/
'''

card_path = os.path.join(PROJECT, 'reports', 'modeling', 'model_card.md')
with open(card_path, 'w') as f:
    f.write(card_lines)
print('model_card.md written:', card_path)


model_card.md written: /Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm/reports/modeling/model_card.md


**Output:** `model_card.md` written. It covers what the model predicts, the data it used, test performance in plain language, the top drivers, what the model cannot do, and known weaknesses.

In [31]:

# Final file inventory
import os
def check(p):
    full = os.path.join(PROJECT, p)
    size = os.path.getsize(full) if os.path.exists(full) else 0
    status = 'OK' if size > 0 else 'MISSING/EMPTY'
    print(f'  {status:12s} {p} ({size:,} bytes)')

print('=== File inventory ===')
check('reports/modeling/model_card.md')
check('reports/modeling/tuning.md')
check('reports/modeling/features.md')
check('reports/modeling/features.parquet')
check('reports/modeling/scores.csv')
check('reports/modeling/tables/model_benchmark.csv')
check('reports/modeling/tables/full_benchmark.csv')
check('reports/modeling/tables/decile_lift.csv')
check('reports/modeling/tables/threshold_table.csv')
check('reports/modeling/tables/permutation_importance.csv')
check('reports/modeling/tables/shap_importance.csv')
check('reports/modeling/tables/test_metrics.csv')
check('reports/modeling/charts/feature_vs_revenue.png')
check('reports/modeling/charts/model_comparison.png')
check('reports/modeling/charts/reliability_curve.png')
check('reports/modeling/charts/decile_lift.png')
check('reports/modeling/charts/driver_importance.png')
check('reports/modeling/charts/response_curves.png')
check('models/mmm_model.joblib')
check('models/optuna_ridge.db')
check('models/optuna_lightgbm.db')
check('models/optuna_xgboost.db')
print('=== END ===')


=== File inventory ===
  OK           reports/modeling/model_card.md (4,408 bytes)
  OK           reports/modeling/tuning.md (2,220 bytes)
  OK           reports/modeling/features.md (2,792 bytes)
  OK           reports/modeling/features.parquet (55,299 bytes)
  OK           reports/modeling/scores.csv (18,668 bytes)
  OK           reports/modeling/tables/model_benchmark.csv (226 bytes)
  OK           reports/modeling/tables/full_benchmark.csv (297 bytes)
  OK           reports/modeling/tables/decile_lift.csv (1,030 bytes)
  OK           reports/modeling/tables/threshold_table.csv (350 bytes)
  OK           reports/modeling/tables/permutation_importance.csv (737 bytes)
  OK           reports/modeling/tables/shap_importance.csv (465 bytes)
  OK           reports/modeling/tables/test_metrics.csv (112 bytes)
  OK           reports/modeling/charts/feature_vs_revenue.png (314,606 bytes)
  OK           reports/modeling/charts/model_comparison.png (29,343 bytes)
  OK           reports/modelin